In [ ]:

import random

def criar_baralho(tipo_baralho='sujo'):
    naipes = ['Copas', 'Espadas', 'Ouros', 'Paus']
    valores_comuns = ['4', '5', '6', '7', 'Q', 'J', 'K', 'A', '2', '3']

    baralho = []
    if tipo_baralho == 'sujo':
        # Baralho sujo (40 cartas): exclui 8, 9, 10 e coringas
        for naipe in naipes:
            for valor in valores_comuns:
                baralho.append((valor, naipe))
    elif tipo_baralho == 'limpo':
        # Baralho limpo (24 cartas): exclui 4, 5, 6, 7, 8, 9, 10 e coringas
        valores_limpo = ['Q', 'J', 'K', 'A', '2', '3']
        for naipe in naipes:
            for valor in valores_limpo:
                baralho.append((valor, naipe))
    else:
        raise ValueError("Tipo de baralho inválido. Use 'sujo' ou 'limpo'.")

    return baralho

def definir_manilhas(vira_card):
    # Ordem de força das cartas (sem manilhas): 4, 5, 6, 7, Q, J, K, A, 2, 3
    # A vira define a próxima carta como manilha
    ordem_forca = ['4', '5', '6', '7', 'Q', 'J', 'K', 'A', '2', '3']

    vira_valor = vira_card[0]

    try:
        idx_vira = ordem_forca.index(vira_valor)
        # A manilha é a próxima carta na ordem de força
        manilha_valor = ordem_forca[(idx_vira + 1) % len(ordem_forca)]
    except ValueError:
        # Se a vira for um 3, a manilha é o 4 (ciclo)
        if vira_valor == '3':
            manilha_valor = '4'
        else:
            raise ValueError(f"Valor da vira inválido: {vira_valor}")

    # Ordem de força das manilhas: Paus > Copas > Espadas > Ouros
    manilhas = {
        'Paus': (manilha_valor, 'Paus'),
        'Copas': (manilha_valor, 'Copas'),
        'Espadas': (manilha_valor, 'Espadas'),
        'Ouros': (manilha_valor, 'Ouros')
    }

    # O ZAP é a manilha de Paus
    zap = manilhas['Paus']

    return manilhas, zap

def simular_rodada(num_jogadores, tipo_baralho='sujo'):
    baralho = criar_baralho(tipo_baralho)
    random.shuffle(baralho)

    # Distribuir 3 cartas para cada jogador
    cartas_jogadores = []
    for _ in range(num_jogadores):
        cartas_jogador = [baralho.pop() for _ in range(3)]
        cartas_jogadores.append(cartas_jogador)

    # A vira é a próxima carta do baralho
    if not baralho:
        raise ValueError("Baralho vazio antes de virar a carta. Verifique o número de jogadores e o tipo de baralho.")

    vira_card = baralho.pop(0)

    _, zap_card = definir_manilhas(vira_card)

    # Verificar se algum jogador tem o ZAP
    jogador_com_zap = False
    for i, cartas in enumerate(cartas_jogadores):
        if zap_card in cartas:
            jogador_com_zap = True
            break

    return jogador_com_zap

def simular_monte_carlo(num_simulacoes, num_jogadores, tipo_baralho):
    contagem_zap = 0
    for _ in range(num_simulacoes):
        try:
            if simular_rodada(num_jogadores, tipo_baralho):
                contagem_zap += 1
        except ValueError as e:
            # print(f"Erro na simulação: {e}. Pulando esta rodada.") # Comentado para reduzir poluição na saída
            continue

    probabilidade = contagem_zap / num_simulacoes
    return probabilidade

if __name__ == '__main__':
    print("--- Simulação de Monte Carlo ---")
    num_simulacoes = 100000

    cenarios = {
        "Individual": 2,
        "Dupla": 4,
        "Trio": 6
    }

    tipos_baralho = ["sujo", "limpo"]

    resultados = {}

    for tipo_baralho in tipos_baralho:
        for cenario, num_jogadores in cenarios.items():
            print(f"Simulando {cenario} com baralho {tipo_baralho}...")
            prob = simular_monte_carlo(num_simulacoes, num_jogadores, tipo_baralho)
            resultados[f"{cenario} - {tipo_baralho}"] = prob

    print("\nResultados Finais:")
    for cenario_baralho, prob in resultados.items():
        print(f"{cenario_baralho}: {prob*100:.2f}%") # Alterado para 2 casas decimais




--- Simulação de Monte Carlo ---
Simulando Individual com baralho sujo...
Simulando Dupla com baralho sujo...
Simulando Trio com baralho sujo...
Simulando Individual com baralho limpo...
Simulando Dupla com baralho limpo...
Simulando Trio com baralho limpo...

Resultados Finais:
Individual - sujo: 15.20%
Dupla - sujo: 31.07%
Trio - sujo: 46.24%
Individual - limpo: 21.63%
Dupla - limpo: 43.41%
Trio - limpo: 65.28%
